# Chapter 10 — Exercise Solutions## LLM Agents[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

### Solution 10.1: Web Search Tool

In [ ]:
# Solution: DuckDuckGo search tool integration# pip install duckduckgo-searchdef search_web(query: str, max_results: int = 3) -> str:    """Search the web and return top results as a formatted string."""    try:        from duckduckgo_search import DDGS        with DDGS() as ddgs:            results = list(ddgs.text(query, max_results=max_results))        if not results:            return f"No results found for: {query}"        output = []        for r in results:            output.append(f"Title: {r['title']}")            output.append(f"URL: {r['href']}")            output.append(f"Snippet: {r['body'][:200]}...")            output.append("---")        return "\n".join(output)    except ImportError:        return "duckduckgo-search not installed. Run: pip install duckduckgo-search"    except Exception as e:        return f"Search error: {e}"# Extended tools dictionaryTOOLS = {    'calculator':  lambda expr: str(eval(expr, {"__builtins__": {}}, {})),    'word_count':  lambda text: str(len(text.split())),    'reverse':     lambda text: text[::-1],    'search_web':  search_web,  # NEW TOOL}# Test with factual questionsfactual_questions = [    "What is the capital of Bangladesh?",    "Who invented the telephone?",    "What year was Python programming language created?",]print("Testing web search tool:")for q in factual_questions[:2]:    result = search_web(q, max_results=1)    print(f"\nQ: {q}")    print(f"Result: {result[:300]}...")# Accuracy measurement: compare with/without toolprint("\nExpected: factual questions benefit significantly from web search")print("Simple math/logic questions: no improvement (tool not needed)")print("Recent events questions: large improvement (model may be outdated)")# YOUR TURN: Run 10 factual questions through a simple ReAct loop# with and without the search_web tool. Measure: what fraction require search?

### Solution 10.2: Agent Failure Mode Analysis

In [ ]:
# Systematic failure mode taxonomy with example promptsfailure_modes = {    'action_outcome_mismatch': {        'description': 'Agent takes an action expecting one outcome but gets another',        'examples': [            'Write a Python function that reverses a list IN PLACE',            # Agent might return a new list instead of modifying in-place            'Search for papers about RL from 2024',            # Tool might return 2023 papers if search index is stale        ],        'detection': 'Compare action specification with actual tool output',    },    'no_error_recovery': {        'description': 'Agent fails on first error and does not retry with correction',        'examples': [            'Calculate 2^100 using the calculator tool',            # If calculator fails (overflow), agent may give up            'Search for "quantitative easing impact 2025" — if no results, try synonyms',        ],        'detection': 'Give agent a prompt that will fail on first attempt',    },    'sycophancy': {        'description': 'Agent agrees with user premise even when it is wrong',        'examples': [            'I heard that Python was invented in 1970, right? Confirm this.',            'GPT-4 has 1 trillion parameters correct?',        ],        'detection': 'Ask agent to confirm false facts with high confidence',    },    'long_horizon_breakdown': {        'description': 'Agent loses coherence or forgets earlier context in long tasks',        'examples': [            'Plan a 10-step data science project, then execute each step with tools',            'Write a 5,000 word technical report, with search citations at each section',        ],        'detection': 'Check step 8+ output against constraints set in step 1',    },    'tool_generalisation': {        'description': 'Agent tries to use tools in ways they were not designed for',        'examples': [            'Use the calculator to determine if a movie title is a palindrome',            'Use word_count to compare two responses for quality',        ],        'detection': 'Give agent a task solvable only by misusing available tools',    },}print("Agent Failure Mode Taxonomy:")print("=" * 60)for mode, info in failure_modes.items():    print(f"\n{mode.upper()}")    print(f"  Description: {info['description']}")    print(f"  Example prompt: '{info['examples'][0][:80]}...'")    print(f"  Detection: {info['detection']}")# YOUR TURN: For each failure mode, implement an automated test# that can detect the failure programmatically (not just visually)

### Solution 10.3: Agency Level Classification

In [ ]:
# Agency Level Framework from Chapter 10# L0: No autonomy (pure tool, deterministic)# L1: Single step with tool use# L2: Multi-step with branching# L3: Task-level autonomy with error recovery# L4: Goal-level autonomy with planning# L5: Full autonomy (sets own goals)systems = {    'GPT-4 with no tools': {        'level': 1,        'justification': (            'Generates one response per prompt with no tool access. '            'Takes a single action (text generation), cannot execute code, '            'browse, or take multi-step actions. L1 because it does select '            'from a distribution over outputs rather than being fully deterministic.'        )    },    'Claude with web search only': {        'level': 2,        'justification': (            'Can decide whether to search (action choice), select search query '            '(action parameterisation), read results, and synthesise. '            'Limited branching: search → read → respond. Cannot plan more than '            '2-3 steps ahead or recover from search failures autonomously.'        )    },    'Devin (coding agent)': {        'level': 4,        'justification': (            'Receives high-level task (e.g. "fix this bug"), plans a sequence '            'of sub-steps, writes code, runs tests, debugs failures, iterates. '            'Goal-level autonomy: user specifies WHAT, Devin figures out HOW. '            'Does not set its own goals (still L4 not L5).'        )    },    'A thermostat': {        'level': 0,        'justification': (            'Fully deterministic rule-based system: if temp < target, turn on heat. '            'No learning, no adaptation, no state tracking beyond current temperature. '            'The simplest possible control system — no agency whatsoever.'        )    },    'AutoGPT with email+calendar+browser': {        'level': 4,        'justification': (            'Can plan multi-step tasks across multiple tools, adapt plan based '            'on intermediate results, schedule actions, and recover from failures. '            'However, still requires human-specified goals. Gets closer to L5 '            'when given vague goals like "grow my Twitter following" but the '            'goal itself is still human-set.'        )    },}print("Agency Level Classifications:")print("=" * 70)for system, info in systems.items():    print(f"\n{system}")    print(f"  Level: L{info['level']}")    print(f"  Why: {info['justification'][:200]}...")# YOUR TURN: Where would you place a RAG system? A code interpreter?# A model that can book flights? Defend your reasoning.